<a href="https://colab.research.google.com/github/MarcAmil30/DOGMA/blob/main/esm2_proto_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install git+https://github.com/evo-design/proto-tools.git

  Cloning https://github.com/evo-design/proto-tools.git to /tmp/pip-req-build-bh24p3q2
  Running command git clone --filter=blob:none --quiet https://github.com/evo-design/proto-tools.git /tmp/pip-req-build-bh24p3q2
  Resolved https://github.com/evo-design/proto-tools.git to commit 98273909f1110545b768b681fa1093f96821e746
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 127.9 MB/s eta 0:

In [2]:
import pandas as pd
import numpy as np
import requests

In [3]:
def get_full_seq(transcript):
    transcript = transcript.split(".")[0]   # drop version if present
    url = f"https://rest.ensembl.org/sequence/id/{transcript}"
    r = requests.get(url, params={"type": "protein"},
                     headers={"Content-Type": "application/json"})
    r.raise_for_status()
    return r.json()["seq"]

def get_protein_info(fn, i):
    """
    Get the protein sequence before & after the mutation.
    Handles single-residue and multi-residue (block) changes.
    input:
        fn: file name (VEP tab-delimited output)
        i:  the row index clicked by the user
    output:
        protein_before: protein sequence before the mutation
        protein_after:  protein sequence after the mutation
    """
    df = pd.read_csv(fn, sep="\t")
    aa = df.loc[i, "Amino_acids"]

    # bail out if this row has no amino-acid change
    if not isinstance(aa, str) or "/" not in aa:
        return "", ""

    aa_ref, aa_alt = aa.split("/")
    ref_block = "" if aa_ref == "-" else aa_ref      # "-" means empty (indel)
    alt_block = "" if aa_alt == "-" else aa_alt

    # Protein_position is "711" (single) or "1787-1788" (range), 1-based inclusive
    parts = str(df.loc[i, "Protein_position"]).split("-")
    start = int(parts[0])
    end   = int(parts[-1])          # equals start when it's a single position

    transcript = df.loc[i, "Feature"]
    full_seq = get_full_seq(transcript)

    idx0 = start - 1                # 0-based start
    observed = full_seq[idx0:end]  # the residues the table refers to
    assert observed == ref_block, (
        f"reference mismatch: sequence has {observed!r}, table says {ref_block!r}"
    )

    protein_before = full_seq
    protein_after  = full_seq[:idx0] + alt_block + full_seq[end:]
    return protein_before, protein_after

In [4]:
fn="https://raw.githubusercontent.com/MarcAmil30/DOGMA/refs/heads/main/smxArJipt0KSbaYk.txt"

In [5]:
tseq_wt,tseq_mut=get_protein_info(fn, 0)

In [7]:
tseq_mut

'MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFSQLVEELLKIICAFQLDTGLEYANSYNFAKKENNSPEHLKDEVSIIQSMGYRNRAKRLLQSEPENPSLQETSLSVQLSNLGTVRTLRTKQRIQPQKTSVYIELGSDSSEDTVNKATYCSVGDQELLQITPQGTRDEISLDSAKKGEAASGCESETSVSEDCSGLSSQSDILTTQQRDTMQHNLIKLQQEMAELEAVLEQHGSQPSNSYPSIISDSSALEDLRNPEQSTSEKAVLTSQKSSEYPISQNPEGLSADKFEVSADSSTSKNKEPGVERSSPSKCPSLDDRWYMHSCSGSLQNRNYPSQEELIKVVDVEEQQLEESGPHDLTETSYLPRQDLEGTPYLESGISLFSDDPESDPSEDRAPESARVGNIPSSTSALKVPQLKVAESAQSPAAAHTTDTAGYNAMEESVSREKPELTASTERVNKRMSMVVSGLTPEEFMLVYKFARKHHITLTNLITEETTHVVMKTDAEFVCERTLKYFLGIAGGKWVVSYFWVTQSIKERKMLNEHDFEVRGDVVNGRNHQGPKRARESQDRKIFRGLEICCYGPFTNMPTDQLEWMVQLCGASVVKELSSFTLGTGVHPIVVVQPDAWTEDNGFHAIGQMCEAPVVTREWVLDSVALYQCQELDTCLIPQIPHSHY'

In [8]:
from proto_tools import (
    ESM2ScoringConfig,
    ESM2ScoringInput,
    run_esm2_score,
)

In [9]:
def score_variants (wt,mut):
  inputs = ESM2ScoringInput(sequences=[wt, mut])
  config = ESM2ScoringConfig(
      batch_size=5,
      return_logits=False,
      device="cuda",  # Change to "cpu" if no GPU available
  )
  score_result = run_esm2_score(inputs, config)

  return score_result

  return score_result['perplexity']

In [11]:
result=score_variants(tseq_wt,tseq_mut)

Streaming output truncated to the last 5000 lines.


In [17]:
result.scores[0].perplexity

6.873336987220442

In [20]:
def print_perplexcity_change (result):
  wt=result.scores[0].perplexity
  mut=result.scores[0].perplexity
  print (f"perplexity WT: {wt:.5f}")
  print (f"perplexity MUT: {mut:.5f}")
  print (f"perplexity diff: {mut:.5f}")

In [21]:
print_perplexcity_change (result)

perplexity WT: 6.87334
perplexity MUT: 6.87334
perplexity diff: 6.87334
